# 04 - feature engineering (sample)

mfcc + mel spectrogram extraction on sample data.

In [ ]:
import pandas as pd
import numpy as np
import librosa
from pathlib import Path
from src.config.settings import SAMPLE_RATE, N_MELS, N_MFCC, N_FFT, HOP_LENGTH

SAMPLES = Path("../../data/samples")
df = pd.read_csv(SAMPLES / "sample_labels.csv")

In [ ]:
def extract_mel(fp):
    y, sr = librosa.load(fp, sr=SAMPLE_RATE)
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    return librosa.power_to_db(mel, ref=np.max).T

def extract_mfcc(fp):
    y, sr = librosa.load(fp, sr=SAMPLE_RATE)
    m = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    d = librosa.feature.delta(m)
    d2 = librosa.feature.delta(m, order=2)
    return np.concatenate([m.mean(1), m.std(1), d.mean(1), d.std(1), d2.mean(1), d2.std(1)])

In [ ]:
speech = df[df['channel']=='speech']
print(f"extracting mel for {len(speech)} speech files...")
mels = np.array([extract_mel(fp) for fp in speech['filepath']])
print(f"mel shape: {mels.shape}")

print(f"extracting mfcc for {len(speech)} speech files...")
mfccs = np.array([extract_mfcc(fp) for fp in speech['filepath']])
print(f"mfcc shape: {mfccs.shape}")

features extracted. mel for cnn input, mfcc for svm baseline.